# 🚀 End-to-End AWS SageMaker Pipelines Walkthrough

This interactive notebook demonstrates how to construct, compile, and execute an enterprise
**SageMaker Pipeline DAG** with automated quality gating and Model Registry registration.

### Pipeline Steps:
1. **ProcessingStep**: Preprocessing & Stratified Data Splitting (`src/processing/preprocess.py`)
2. **TrainingStep**: Managed Model Training via SKLearn Estimator (`src/training/train.py`)
3. **ProcessingStep**: Model Evaluation on holdout test set (`src/evaluation/evaluate.py`)
4. **ConditionStep**: Quality Gate (Holdout Accuracy >= 0.85)
5. **ModelStep**: Automated Model Registry Registration (`MobilePriceClassificationModelPackageGroup`)

In [ ]:
import os
import sys
import json
from pathlib import Path

# Add root directory to sys.path
sys.path.insert(0, str(Path(os.getcwd()).parent))

import boto3
import sagemaker
from src.config import load_app_config, load_model_config
from src.pipeline.sagemaker_pipeline import get_pipeline

app_cfg = load_app_config()
model_cfg = load_model_config()
print("App Config Loaded. Target S3 Bucket:", app_cfg.aws.s3_bucket)

## 1. Local Pipeline Simulation (Offline Verification)
Run the pipeline steps sequentially on local compute without requiring AWS credentials.

In [ ]:
from src.pipeline.run_pipeline import run_local_simulation

# Run local end-to-end simulation
run_local_simulation()

## 2. Compile SageMaker Pipeline DAG into JSON Definition
Inspect the compiled AST JSON definition that SageMaker executes.

In [ ]:
from src.pipeline.run_pipeline import build_pipeline_definition

definition = build_pipeline_definition(output_file="../artifacts/pipeline_definition.json")
print("Total DAG Steps:", len(definition["Steps"]))
for step in definition["Steps"]:
    print(f"  Step: {step[Name]} | Type: {step[Type]}")

## 3. SageMaker Inference Handler Verification
Test `model_fn`, `input_fn`, `predict_fn`, and `output_fn` locally.

In [ ]:
from src.inference.inference import model_fn, input_fn, predict_fn, output_fn

# Load model
bundle = model_fn("../artifacts/model")

# Sample payload
sample_input = [{
    "battery_power": 1821, "blue": 1, "clock_speed": 1.2, "dual_sim": 0,
    "fc": 13, "four_g": 1, "int_memory": 44, "m_dep": 0.8, "mobile_wt": 141,
    "n_cores": 6, "pc": 14, "px_height": 1208, "px_width": 1212, "ram": 3911,
    "sc_h": 8, "sc_w": 6, "talk_time": 15, "three_g": 1, "touch_screen": 1, "wifi": 0
}]

df_in = input_fn(sample_input, "application/json")
preds = predict_fn(df_in, bundle)
print("Prediction Response:\n", output_fn(preds, "application/json"))

## 4. Upsert & Execute on AWS SageMaker (Optional - Requires AWS Role)
Uncomment the cell below when AWS credentials and `SAGEMAKER_ROLE_ARN` are configured.

In [ ]:
# from src.pipeline.run_pipeline import upsert_and_execute_pipeline
# execution = upsert_and_execute_pipeline(execute=True)
# print("Execution ARN:", execution.arn)